# EDA: Explainable Grammatical Error Correction Dataset

**Репозиторий:** [lorafei/Explainable_GEC](https://github.com/lorafei/Explainable_GEC)  
**Задача:** Grammatical Error Correction (GEC) с объяснимой коррекцией — каждый пример содержит ошибочное предложение, исправленное предложение, тип ошибки, индексы коррекции и evidence-индексы.

---

## 1. Установка и импорт библиотек

In [ ]:
!pip install -q pandas ujson matplotlib seaborn

In [ ]:
import pandas as pd
import ujson as json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

## 2. Загрузка датасета

In [ ]:
!git clone https://github.com/lorafei/Explainable_GEC.git 2>/dev/null || echo 'already cloned'

In [ ]:
DATA_DIR = Path("Explainable_GEC/data/json")

def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

df_train = load_jsonl(DATA_DIR / "train.json")
df_dev   = load_jsonl(DATA_DIR / "dev.json")
df_test  = load_jsonl(DATA_DIR / "test.json")

# Пометим сплиты и объединим для общего анализа
df_train["split"] = "train"
df_dev["split"]   = "dev"
df_test["split"]  = "test"

df = pd.concat([df_train, df_dev, df_test], ignore_index=True)

print(f"train: {len(df_train):>6,}")
print(f"dev:   {len(df_dev):>6,}")
print(f"test:  {len(df_test):>6,}")
print(f"total: {len(df):>6,}")

## 3. Первичный осмотр

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
# Пример записи
row = df.iloc[0]
print("Source:", " ".join(row["source"]))
print("Target:", " ".join(row["target"]))
print("Error type:", row["error_type"])
print("Correction indices:", row["correction_index"])
print("Evidence indices:", row["evidence_index"])
print("Origin:", row["origin"])

## 4. Feature engineering

In [ ]:
df["source_len"] = df["source"].apply(len)
df["target_len"] = df["target"].apply(len)
df["len_diff"]   = df["target_len"] - df["source_len"]
df["n_corrections"] = df["correction_index"].apply(len)
df["n_evidence"]    = df["evidence_index"].apply(len)

# Текстовые версии предложений
df["source_text"] = df["source"].apply(lambda t: " ".join(t))
df["target_text"] = df["target"].apply(lambda t: " ".join(t))

df[["source_len", "target_len", "len_diff", "n_corrections", "n_evidence"]].describe().round(2)

## 5. Типы ошибок

In [ ]:
print(f"Всего категорий ошибок: {df['error_type'].nunique()}\n")

error_counts = df["error_type"].value_counts()
error_pct    = (error_counts / len(df) * 100).round(1)

pd.DataFrame({"count": error_counts, "%": error_pct})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=error_counts.values, y=error_counts.index, palette="viridis", ax=ax)
ax.set_xlabel("Количество примеров")
ax.set_ylabel("")
ax.set_title("Распределение типов грамматических ошибок")
for i, v in enumerate(error_counts.values):
    ax.text(v + 50, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Распределение по origin (уровень владения)

In [ ]:
origin_counts = df["origin"].value_counts().sort_index()
print(origin_counts)

fig, ax = plt.subplots(figsize=(5, 3))
origin_counts.plot.bar(color=sns.color_palette("viridis", 3), ax=ax)
ax.set_title("Количество примеров по уровню (origin)")
ax.set_ylabel("Count")
ax.set_xlabel("Origin")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Типы ошибок × Origin

In [ ]:
ct = pd.crosstab(df["error_type"], df["origin"], normalize="columns").round(3) * 100
ct = ct.loc[error_counts.index]  # сортируем по частоте

fig, ax = plt.subplots(figsize=(10, 6))
ct.plot.barh(ax=ax, width=0.8)
ax.set_xlabel("% от примеров данного origin")
ax.set_title("Профиль ошибок по уровню владения")
ax.legend(title="Origin")
plt.tight_layout()
plt.show()

## 8. Длины предложений

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["source_len"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Длина source (токены)")
axes[0].set_xlabel("Tokens")

sns.histplot(df["len_diff"], bins=30, kde=True, ax=axes[1], color="salmon")
axes[1].set_title("Разница target − source (токены)")
axes[1].set_xlabel("Δ tokens")
axes[1].axvline(0, color="black", ls="--", lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Средние длины по типу ошибки
len_by_error = (
    df.groupby("error_type")[["source_len", "target_len"]]
      .agg(["mean", "median"])
      .sort_values(("source_len", "mean"), ascending=False)
      .round(1)
)
len_by_error

In [ ]:
# Средние длины по origin
len_by_origin = (
    df.groupby("origin")[["source_len", "target_len"]]
      .agg(["mean", "median"])
      .sort_index()
      .round(1)
)
len_by_origin

## 9. Количество коррекций и evidence-индексов

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["n_corrections"].value_counts().sort_index().plot.bar(ax=axes[0], color="teal")
axes[0].set_title("Число correction indices на пример")
axes[0].set_xlabel("n_corrections")

ev_vc = df["n_evidence"].value_counts().sort_index()
ev_vc[ev_vc.index <= 20].plot.bar(ax=axes[1], color="darkorange")
axes[1].set_title("Число evidence indices на пример")
axes[1].set_xlabel("n_evidence")

plt.tight_layout()
plt.show()

print(f"Примеры без evidence: {(df['n_evidence'] == 0).sum()} ({(df['n_evidence'] == 0).mean()*100:.1f}%)")

## 10. Сравнение сплитов (train / dev / test)

In [ ]:
split_summary = (
    df.groupby("split")
      .agg(
          n=("error_type", "size"),
          n_error_types=("error_type", "nunique"),
          avg_src_len=("source_len", "mean"),
          avg_tgt_len=("target_len", "mean"),
          avg_corrections=("n_corrections", "mean"),
      )
      .round(2)
)
split_summary

In [ ]:
# Распределение ошибок одинаково по сплитам?
ct_split = pd.crosstab(df["error_type"], df["split"], normalize="columns").round(3) * 100
ct_split = ct_split.loc[error_counts.index]

fig, ax = plt.subplots(figsize=(10, 6))
ct_split[["train", "dev", "test"]].plot.barh(ax=ax, width=0.8)
ax.set_xlabel("% от сплита")
ax.set_title("Распределение типов ошибок по сплитам")
ax.legend(title="Split")
plt.tight_layout()
plt.show()

## 11. Токен `[NONE]` — вставки и удаления

In [ ]:
df["none_in_source"] = df["source"].apply(lambda t: t.count("[NONE]"))
df["none_in_target"] = df["target"].apply(lambda t: t.count("[NONE]"))

print("[NONE] в source (= нужна вставка):")
print(df["none_in_source"].value_counts().sort_index().head(5))
print(f"\nДоля примеров с [NONE] в source: {(df['none_in_source'] > 0).mean()*100:.1f}%")

print("\n[NONE] в target (= нужно удаление):")
print(df["none_in_target"].value_counts().sort_index().head(5))
print(f"\nДоля примеров с [NONE] в target: {(df['none_in_target'] > 0).mean()*100:.1f}%")

## 12. Корреляции числовых признаков

In [ ]:
num_cols = ["source_len", "target_len", "len_diff", "n_corrections", "n_evidence",
            "none_in_source", "none_in_target"]

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[num_cols].corr().round(2), annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Матрица корреляций")
plt.tight_layout()
plt.show()

## 13. Дополнительные данные: gector.json и t5.json

In [ ]:
df_gector = load_jsonl(DATA_DIR / "gector.json")
df_t5     = load_jsonl(DATA_DIR / "t5.json")

print(f"gector.json: {len(df_gector)} samples, columns: {list(df_gector.columns)}")
print(f"t5.json:     {len(df_t5)} samples, columns: {list(df_t5.columns)}")

print("\ngector sample:")
display(df_gector.head(2))
print("\nt5 sample:")
display(df_t5.head(2))

## 14. Выводы

| Характеристика | Значение |
|---|---|
| Общее число примеров | 20 016 (train 15 187 / dev 2 413 / test 2 416) |
| Категорий ошибок | 15 |
| Топ-3 ошибки | Preposition, Verb Tense, Number |
| Средняя длина предложения | ~31 токен |
| Уровни владения (origin) | A (6 926), B (5 944), C (2 317) |
| Сплиты сбалансированы по типам ошибок | Да |

Датасет хорошо подходит для задачи генерации персонализированных упражнений, так как содержит:
- типизированные ошибки (15 категорий),
- уровни владения (origin A/B/C),
- объяснимые evidence-индексы для интерпретируемости коррекций.